In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [2]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_789/Mrk-789_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_789/Mrk-789_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     379   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        60   (2094, 964)   float32   


In [3]:
image_cut = image[75:175, 0:1890]
image_error_cut = image_error[75:175, 0:1890]

In [4]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.03144
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [5]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) #+
       # sersic_1d(x_hr, params4)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [6]:
seeing = 3.174
sigma = seeing / 2.35 
x_axes = np.arange(0, 100, 1)

In [7]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, a, b):
    func = model_convolved(x, params1, params2, params3)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    sum_area = blue_area + green_area +purple_area 
    total_area = total_integral(x, params1, params2, params3, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area]     

## Halpha

In [8]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_789/Fit/Halpha_fit.csv", index_col=0)

In [9]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

165.74469214562285
172.02593497188366
43.47727775343067
43.81231499748881
35.74401161829957
35.62311829705922
           Peak 1     Peak 2     Peak 3
Blue    96.357919  10.727478   0.864735
Green    3.628098  85.262858  18.556870
Purple   0.013983   4.009664  80.578394


In [10]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

353.04655802491163
363.8493971477083
82.509837996196
83.03432627123688
67.3786352505155
67.1877231690311
           Peak 1     Peak 2     Peak 3
Blue    95.948079  12.169970   0.965547
Green    4.036217  83.177313  20.114758
Purple   0.015704   4.652718  78.919695


In [11]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

428.11634220911304
439.065813938423
135.84857942535297
136.40125051024424
101.97830107139347
101.7778373943939
           Peak 1     Peak 2     Peak 3
Blue    95.133431  19.258093   1.091541
Green    4.844098  75.687204  22.487421
Purple   0.022472   5.054704  76.421038


## HBeta

In [12]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_789/Fit/Hbeta_fit.csv", index_col = 0)

In [13]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

44.51546260059282
44.298767129693154
12.122913933939056
11.901384385414897
10.73403532642865
10.609525504317112
           Peak 1     Peak 2     Peak 3
Blue    95.738477  11.824853   0.245270
Green    4.208535  84.536922  18.084634
Purple   0.052988   3.638225  81.670096


In [14]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

81.3384695354924
80.98689491449593
22.949284805166755
22.599933881282688
19.963875487995036
19.766677081275574
           Peak 1     Peak 2     Peak 3
Blue    95.271506  13.798271   0.289714
Green    4.667604  82.112259  19.715951
Purple   0.060890   4.089471  79.994334


In [15]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


118.18974966634372
117.78571378891422
36.017269157800044
35.62055513583968
29.746076853148228
29.52532234318089
           Peak 1     Peak 2     Peak 3
Blue    93.627665  15.854356   0.348033
Green    6.274580  77.697150  22.444584
Purple   0.097755   6.448494  77.207383


## NII

In [16]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_789/Fit/NII_fit.csv", index_col = 0)

In [17]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

80.39347173597102
80.93484930014543
9.750597173759049
9.960133085873661
14.601526755694344
14.72502037452642
           Peak 1     Peak 2     Peak 3
Blue    94.720424   5.620282   0.016444
Green    5.275415  93.041280  29.612663
Purple   0.004161   1.338438  70.370893


In [18]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

147.19998747515004
148.01191413037947
45.72875522744648
46.40831891470145
32.31842781699051
32.55906231745765
           Peak 1     Peak 2     Peak 3
Blue    94.142859   8.553169   0.019844
Green    5.851979  89.654465  31.774545
Purple   0.005162   1.792366  68.205611


In [19]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

212.08999668001198
212.7801110141193
61.78692982575158
62.48083482026717
41.60310879823274
41.83999452569696
           Peak 1     Peak 2     Peak 3
Blue    93.140190  12.336831   0.033454
Green    6.853111  85.335052  35.626429
Purple   0.006698   2.328117  64.340117


## SII

In [20]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
x_axes = np.arange(0, 100, 1)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_789/Fit/SII_fit.csv", index_col = 0)

In [21]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

33.66217456846005
33.672300762696175
5.157346510683369
5.308616167710817
4.18042825100828
4.244103330018698
           Peak 1     Peak 2     Peak 3
Blue    93.337259   6.917336   0.001332
Green    6.635053  90.405619  24.030157
Purple   0.027688   2.677045  75.968511


In [22]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

63.1992768313332
63.21031032973653
24.321784556824113
24.812608249459103
18.876479730775728
19.081291469234312
           Peak 1     Peak 2     Peak 3
Blue    92.798795  10.953129   0.003835
Green    7.169359  85.758247  27.242397
Purple   0.031845   3.288624  72.753768


In [23]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

103.39677806778404
103.35672618343051
33.19844664901781
33.70237698260303
24.34039479845097
24.548612550352544
           Peak 1     Peak 2     Peak 3
Blue    90.639113  15.807604   0.008917
Green    9.309021  80.267969  30.374622
Purple   0.051866   3.924427  69.616460


## OIII

In [24]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/Mrk_789/Fit/OIII_fit.csv", index_col = 0)

In [25]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -sigma, 
                                                                fit['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -sigma, 
                                                                fit['Component 2'].iloc[3] + sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -sigma, 
                                                                fit['Component 3'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

34.697234476876005
34.781322661322065
10.026948171258379
10.047603404040094
10.815238251234105
10.689646732686619
           Peak 1     Peak 2     Peak 3
Blue    92.891855   6.253405   0.002255
Green    7.063229  91.090077  21.193354
Purple   0.044916   2.656517  78.804391


In [26]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -2*sigma, 
                                                                fit['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -2*sigma, 
                                                                fit['Component 2'].iloc[3] + 2*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -2*sigma, 
                                                                fit['Component 3'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

64.3788001013295
64.50464344525577
19.30584899001469
19.33991960258369
19.98320883101294
19.782749840295516
           Peak 1     Peak 2     Peak 3
Blue    92.263079   7.904685   0.003419
Green    7.686301  89.158449  23.143895
Purple   0.050620   2.936866  76.852687


In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 1'].iloc[3] -3*sigma, 
                                                                fit['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 2'].iloc[3] -3*sigma, 
                                                                fit['Component 2'].iloc[3] + 3*sigma)
data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 3'].iloc[3] -3*sigma, 
                                                                fit['Component 3'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(3)],
                  index=['Blue', 'Green', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

96.02683629958332
96.17063948847849
31.219195111057246
31.26145805459026
30.484661913810967
30.240341706345387
           Peak 1     Peak 2     Peak 3
Blue    90.079776   9.833153   0.012650
Green    9.842427  85.658920  28.850607
Purple   0.077797   4.507926  71.136743
